# レッスン13 - Cogneeナレッジグラフを用いたエージェントメモリ


## セットアップ

このノートブックでは、[**Cognee**](https://www.cognee.ai/) のナレッジグラフと **Microsoft Agent Framework** (MAF) を使用して、永続的なメモリを備えたインテリジェントな**コーディングアシスタント**を構築する方法を実演します。

Cognee は、非構造化テキストを、ベクトル埋め込みによって裏打ちされた、クエリ可能な構造化ナレッジグラフに変換します。これにより、エージェントは情報の関連性を理解した、リッチな長期記憶を持つことが可能になります。

### 学べること
1. **ナレッジグラフの構築**: 開発者のプロフィールやベストプラクティスを、クエリ可能な構造化された知識に変換します。
2. **Cognee と MAF の統合**: `@tool` 関数を使用して、MAF エージェントが Cognee のナレッジグラフに対してクエリを実行できるようにします。
3. **セッションを認識した会話**: 同一セッション内での複数の質問にわたってコンテキストを維持します。
4. **長期記憶**: 重要な知識をセッション間で永続化し、新しい会話でそれを取得・活用します。

### 前提条件
- Python 3.12 以上
- セッション管理用にローカルで実行されている Redis (`docker run -d -p 6379:6379 redis`)
- LLM API キー (例: OpenAI) — `.env` ファイルに `LLM_API_KEY` を設定
- `.env` ファイルに `CACHING=true` を設定 (Cognee のセッションに必要)
- チャットモデルがデプロイされた Microsoft Foundry プロジェクト
- `.env` ファイルに `AZURE_AI_PROJECT_ENDPOINT` と `AZURE_AI_MODEL_DEPLOYMENT_NAME` を設定
- Azure CLI での認証済み状態 (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## エージェントメモリの種類

このノートブックでは、メインのレッスン13ノートブックと同じ3種類のメモリを探りますが、長期メモリのバックエンドとしてCogneeを使用します：

| メモリタイプ | メカニズム | 寿命 |
|---|---|---|
| **作業中（ワーキング）** | `agent.create_session()` (MAF) | 単一の会話 |
| <strong>短期</strong> | Cognee セッションキャッシュ（Redis） | 単一のセッション |
| <strong>長期</strong> | Cognee 知識グラフ + ベクトル | 永続的 |

### Cogneeのメモリアーキテクチャ
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Cognee ストレージの準備


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## パート 1 — ナレッジベースの構築

私たちはコーディングアシスタントのために包括的なナレッジベースを作成するために、3種類のデータを取り込みます：

1. <strong>開発者プロフィール</strong> — 個人の専門知識と技術的背景
2. **Pythonのベストプラクティス** — 実践的なガイドラインを伴うPythonの禅
3. <strong>過去の会話</strong> — 開発者とAIアシスタント間の過去のQ&Aセッション


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ナレッジグラフの可視化

Cognee は抽出したエンティティと関係のインタラクティブな HTML ビジュアライゼーションをレンダリングできます。


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memifyで記憶を豊かにする

`memify()` はナレッジグラフを解析し、パターン、ベストプラクティス、概念間の関係を特定するインテリジェントなルールを生成します。


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## パート2 — Cogneeツールを使ったMAFエージェント

ここでは、`@tool` 関数を介してCogneeのナレッジグラフにクエリを実行できるMAFエージェントを作成します。これにより、エージェントはグラフ対応のセマンティック検索の力を最大限に活用しつつ、セッションを通じて会話の文脈を維持できます。


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## セッションによる作業記憶

`AgentSession`（`agent.create_session()`経由で作成）は、セッション内の作業記憶を提供します。エージェントは以前のメッセージに戻って参照しながら、Cogneeの長期ナレッジグラフにも問い合わせることができます。


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## 新しいセッション — 長期記憶は持続

新しいセッションを開始すると作業記憶はクリアされますが、Cogneeナレッジグラフは依然として利用可能です。エージェントは完全に新しい会話でも同じ長期記憶を取得することができます。


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## まとめ

このノートブックでは、**MAFの作業記憶**（`agent.create_session()`）と**Cogneeの長期知識グラフ**を組み合わせたコーディングアシスタントを構築しました。

### 学んだこと
1. **知識グラフ構築**: Cogneeは非構造化テキストを取り込み、グラフとベクトルメモリを構築します。
2. **memifyによるグラフの強化**: 既存のグラフに派生した事実や豊かな関係性を追加します。
3. **MAF + Cogneeの統合**: `@tool`関数により、MAFエージェントが自然にCogneeのグラフにクエリを投げられます。
4. **作業記憶 + 長期記憶**: `AgentSession`（`agent.create_session()`経由）でセッションコンテキストを持ちつつ、Cogneeが永続的な知識を提供します。
5. **NodeSetsによるフィルタ検索**: 知識グラフの特定のサブセットを対象にする（例：原則のみ）。

### 主なポイント
- **Cognee**は生テキストを構造化され関係性を持つメモリに変換し、単純なベクトルストアより強力です。
- **`@tool`関数**はMAFエージェントと外部知識システムをきれいに橋渡しします。
- **`AgentSession`**（`agent.create_session()`経由）は会話ごとのコンテキストを長期的な知識から切り離して管理します。
- 同じ知識グラフが複数のセッションやエージェントで共有されます。

### 実世界での応用例
- **開発者向けコパイロット**：コードレビュー、インシデント分析、アーキテクチャ支援
- **顧客対応コパイロット**：製品ドキュメント、FAQ、CRMノートに基づくサポートエージェント
- **社内専門家コパイロット**：方針、法務、安全保障のガイドラインに基づくアシスタント
- **統合データレイヤー**：構造化データと非構造化データを一つのクエリ可能なグラフに統合

### 次のステップ
- Cogneeで時間認識の実験を行う
- ドメイン固有のグラフ品質のためOWLオントロジーを定義する
- 時間経過で取得精度を改善するためのユーザーフィードバックループを追加する
- 同じCogneeメモリーレイヤーを共有するマルチエージェントシステムに拡張する


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
